In [1]:

import os
os.environ["DEBUG_TWS_CALLBACK"] = "true"
import logging
logging.basicConfig(level=logging.DEBUG, format='%(levelname)s: %(message)s')
from trading_api.providers.tws.tws_connection import TWSClient
from trading_api.providers.tws import TWSProvider
from trading_api.models import QuoteValues
provider = TWSProvider()

In [2]:
results = await provider.search_symbols('AAPL')
results

INFO: Socket connected: True: ('127.0.0.1', 7497)
DEBUG: Server version: 203, Connection time: 20251126 21:03:39 Central European Standard Time
INFO: IBSocket connection successfully.
INFO: IBSocket reader loop started.
DEBUG: managedAccounts, {'accountsList': 'DU6968828'}
DEBUG: nextValidId, {'orderId': 1}
DEBUG: awaiting symbolSamples for reqId 0 and pattern 'AAPL'
ERROR: TWS error [reqId=-1, time=1764187419446, code=2104]: Market data farm connection is OK:usfarm
ERROR: TWS error [reqId=-1, time=1764187419446, code=2104]: Market data farm connection is OK:usfarm
ERROR: TWS error [reqId=-1, time=1764187419447, code=2106]: HMDS data farm connection is OK:ushmds
ERROR: TWS error [reqId=-1, time=1764187419447, code=2106]: HMDS data farm connection is OK:ushmds
ERROR: TWS error [reqId=-1, time=1764187419447, code=2158]: Sec-def data farm connection is OK:secdefil
ERROR: TWS error [reqId=-1, time=1764187419447, code=2158]: Sec-def data farm connection is OK:secdefil
DEBUG: symbolSamples, 

[SearchSymbolResultItem(symbol='AAPL', description='APPLE INC', exchange='NASDAQ', ticker='AAPL', type='stock'),
 SearchSymbolResultItem(symbol='AAPL', description='APPLE INC', exchange='MEXI', ticker='AAPL', type='stock'),
 SearchSymbolResultItem(symbol='AAPL', description='APPLE INC', exchange='EBS', ticker='AAPL', type='stock'),
 SearchSymbolResultItem(symbol='AAPL', description='APPLE INC-CDR', exchange='TSE', ticker='AAPL', type='stock'),
 SearchSymbolResultItem(symbol='', description='Apple Inc', exchange='', ticker='', type='bond'),
 SearchSymbolResultItem(symbol='AAPLUSD', description='APPLE INC', exchange='EBS', ticker='AAPLUSD', type='stock'),
 SearchSymbolResultItem(symbol='', description='Ascendas Singbridge', exchange='', ticker='', type='bond'),
 SearchSymbolResultItem(symbol='', description='Aarush Phase IV Logistics Park Pvt Ltd', exchange='', ticker='', type='bond'),
 SearchSymbolResultItem(symbol='AVSPY', description='Nasdaq OMX Alpha AAPL vs. SPY Index', exchange='NA

In [ ]:
data = await provider.get_symbol_info('AAPL', 'SMART')
data

In [ ]:
from datetime import datetime, timedelta
from trading_api.models.market import TimeFrame
import pytz
# Define time range (last week)
end_time = datetime.now().astimezone(pytz.UTC)
start_time = end_time - timedelta(days=7)

# Request hourly bars for AAPL
bars = await provider.get_historical_bars(
    symbol="AAPL",
    start_time=start_time,
    end_time=end_time,
    resolution=TimeFrame.HOUR_1,
    exchange="SMART",  # Optional - uses smart routing
    timeout=30.0,      # Optional - 30 second timeout
)

# Access the bar data
for bar in bars:
    print(f"Time: {bar.time}, Open: {bar.open}, High: {bar.high}, "
          f"Low: {bar.low}, Close: {bar.close}, Volume: {bar.volume}")

In [3]:
quotes = await provider.get_quotes_snapshot(
    symbols=["AAPL", "GOOGL", "MSFT", "TSLA"],
    exchange="SMART",  # Optional: smart routing (default)
    timeout=4.0       # Optional: 15s timeout (default)
)

# Process results
for quote in quotes:
    if quote.s == "ok" and isinstance (quote.v, QuoteValues):
        v = quote.v
        print(f"{quote.n}: Last=${v.lp:.2f}, "
                f"Bid=${v.bid:.2f}, Ask=${v.ask:.2f}, "
                f"Volume={v.volume:,}, Change={v.chp:+.2f}%")
    else:
        print(f"{quote.n}: Error - {quote.s}")

DEBUG: awaiting tickSnapshotEnd for reqId 1, symbol='AAPL'
DEBUG: awaiting tickSnapshotEnd for reqId 2, symbol='GOOGL'
DEBUG: awaiting tickSnapshotEnd for reqId 3, symbol='MSFT'
DEBUG: awaiting tickSnapshotEnd for reqId 4, symbol='TSLA'
DEBUG: marketDataType, {'reqId': 1, 'marketDataType': 1}
DEBUG: tickReqParams, {'tickerId': 1, 'minTick': 0.01, 'bboExchange': '9c0001', 'snapshotPermissions': 3}
DEBUG: tickString, {'reqId': 1, 'tickType': 45, 'value': '1764187422'}
DEBUG: tickPrice, {'reqId': 1, 'tickType': 4, 'price': 278.79, 'tickAttrib': 129054955592016: CanAutoExecute: 0, PastLimit: 0, PreOpen: 0}
DEBUG: tickSize, {'reqId': 1, 'tickType': 5, 'size': Decimal('100')}
DEBUG: tickGeneric, {'reqId': 1, 'tickType': 49, 'value': 0.0}
DEBUG: tickSize, {'reqId': 1, 'tickType': 8, 'size': Decimal('195600')}
DEBUG: tickPrice, {'reqId': 1, 'tickType': 6, 'price': 279.53, 'tickAttrib': 129054932952656: CanAutoExecute: 0, PastLimit: 0, PreOpen: 0}
DEBUG: tickPrice, {'reqId': 1, 'tickType': 7, '

AAPL: Last=$278.79, Bid=$278.79, Ask=$278.81, Volume=195,600, Change=+0.66%
GOOGL: Last=$319.02, Bid=$319.00, Ask=$319.04, Volume=375,401, Change=-1.37%
MSFT: Last=$486.55, Bid=$486.52, Ask=$486.56, Volume=391,929, Change=+2.00%
TSLA: Last=$424.73, Bid=$424.71, Ask=$424.75, Volume=1,299,638, Change=+1.27%


In [ ]:
del provider